In [16]:
"""
Problem 01: Top-N per group (e.g., top 3 posts per user by engagement).

Meta flavor: "Show the top 5 reels by watch-time per creator in the last 7 days."

How to Think:
- Rank within a partition with ROW_NUMBER() or RANK(), then filter.
- ROW_NUMBER() = exact N per group; RANK() = allow ties (may exceed N).
- Always partition by the group key; order by the metric descending.

How to Remember:
- "PARTITION BY group, ORDER BY metric DESC, then wrap and filter rn <= N."
- ROW_NUMBER breaks ties arbitrarily → use when duplicates are not meaningful.

AI Use Cases:
- Top-k recommendation candidates before reranking.
- Per-user feature extraction (top 3 clicked items → embedding).
- Anomaly detection: top-1 baseline vs current.
"""
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

spark = SparkSession.builder.getOrCreate()

# posts(user_id, post_id, engagement_score, created_at)
data = [
    (1, 101, 50, "2026-01-01"),
    (1, 102, 80, "2026-01-02"),
    (1, 103, 30, "2026-01-03"),
    (2, 201, 90, "2026-01-01"),
    (2, 202, 70, "2026-01-02"),
]
posts = spark.createDataFrame(data, ["user_id", "post_id", "engagement_score", "created_at"])

w = Window.partitionBy("user_id").orderBy(col("engagement_score").desc())
top3 = posts.withColumn("rn", row_number().over(w)).filter(col("rn") <= 3)
top3.show()
posts.createOrReplaceTempView("user_data_vw")
# SQL equivalent (Presto / Hive)
SQL = """
select * from 
(
select user_id,post_id,engagement_score,created_at , ROW_NUMBER() OVER (partition by user_id order by  engagement_score DESC ) as rn_num 
FROM user_data_vw
) t
WHERE rn_num <= 1;
"""

posts.createOrReplaceTempView("user_data_vw")

+-------+-------+----------------+----------+---+
|user_id|post_id|engagement_score|created_at| rn|
+-------+-------+----------------+----------+---+
|      1|    102|              80|2026-01-02|  1|
|      1|    101|              50|2026-01-01|  2|
|      1|    103|              30|2026-01-03|  3|
|      2|    201|              90|2026-01-01|  1|
|      2|    202|              70|2026-01-02|  2|
+-------+-------+----------------+----------+---+



In [17]:
spark.sql(SQL).show()

+-------+-------+----------------+----------+------+
|user_id|post_id|engagement_score|created_at|rn_num|
+-------+-------+----------------+----------+------+
|      1|    102|              80|2026-01-02|     1|
|      2|    201|              90|2026-01-01|     1|
+-------+-------+----------------+----------+------+



In [8]:
spark.sql("select * from user_data_vw").show()

+-------+-------+----------------+----------+
|user_id|post_id|engagement_score|created_at|
+-------+-------+----------------+----------+
|      1|    101|              50|2026-01-01|
|      1|    102|              80|2026-01-02|
|      1|    103|              30|2026-01-03|
|      2|    201|              90|2026-01-01|
|      2|    202|              70|2026-01-02|
+-------+-------+----------------+----------+



In [18]:
"""
Problem 02: Compare current session to previous session using LAG / LEAD.

Meta flavor: "For each user, compute the time delta between consecutive sessions."

How to Think:
- LAG(col, 1) reads the previous row's value within the same partition.
- LEAD(col, 1) reads the next row's value.
- Order by event_time per user to define "consecutive."

How to Remember:
- "LAG = look BACK, LEAD = look FORWARD."
- Always order inside the window or it's meaningless.

AI Use Cases:
- Sequential feature engineering (delta time → RNN input).
- Churn prediction: gap between sessions.
- Drop-off detection: long delays before next step.
"""
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag, unix_timestamp

spark = SparkSession.builder.getOrCreate()

data = [
    (1, "session_start", "2026-01-01 10:00:00"),
    (1, "session_start", "2026-01-01 14:30:00"),
    (1, "session_start", "2026-01-03 09:00:00"),
    (2, "session_start", "2026-01-02 11:00:00"),
]
events = spark.createDataFrame(data, ["user_id", "event", "ts"])

w = Window.partitionBy("user_id").orderBy("ts")
result = events.withColumn("prev_ts", lag("ts", 1).over(w))
result = result.withColumn(
    "delta_hours",
    (unix_timestamp("ts") - unix_timestamp("prev_ts")) / 3600
)
result.show(truncate=False)

SQL = """
SELECT user_id, event, ts,
       LAG(ts, 1) OVER (PARTITION BY user_id ORDER BY ts) AS prev_ts,
       UNIX_TIMESTAMP(ts) - UNIX_TIMESTAMP(LAG(ts, 1) OVER (PARTITION BY user_id ORDER BY ts)) AS delta_sec
FROM events;
"""


+-------+-------------+-------------------+-------------------+-----------+
|user_id|event        |ts                 |prev_ts            |delta_hours|
+-------+-------------+-------------------+-------------------+-----------+
|1      |session_start|2026-01-01 10:00:00|NULL               |NULL       |
|1      |session_start|2026-01-01 14:30:00|2026-01-01 10:00:00|4.5        |
|1      |session_start|2026-01-03 09:00:00|2026-01-01 14:30:00|42.5       |
|2      |session_start|2026-01-02 11:00:00|NULL               |NULL       |
+-------+-------------+-------------------+-------------------+-----------+



In [19]:
events.createOrReplaceTempView("event_vw")

In [20]:
spark.sql("select * from event_vw").show()

+-------+-------------+-------------------+
|user_id|        event|                 ts|
+-------+-------------+-------------------+
|      1|session_start|2026-01-01 10:00:00|
|      1|session_start|2026-01-01 14:30:00|
|      1|session_start|2026-01-03 09:00:00|
|      2|session_start|2026-01-02 11:00:00|
+-------+-------------+-------------------+



In [28]:
SQL = """
SELECT user_id, event, ts,
       LAG(ts, 1) OVER (PARTITION BY user_id ORDER BY ts) AS prev_ts,
       (UNIX_TIMESTAMP(ts) - UNIX_TIMESTAMP(LAG(ts, 1) OVER (PARTITION BY user_id ORDER BY ts)))/(60*60) AS delta_sec
FROM event_vw;
"""


In [29]:
spark.sql(SQL).show()

+-------+-------------+-------------------+-------------------+---------+
|user_id|        event|                 ts|            prev_ts|delta_sec|
+-------+-------------+-------------------+-------------------+---------+
|      1|session_start|2026-01-01 10:00:00|               NULL|     NULL|
|      1|session_start|2026-01-01 14:30:00|2026-01-01 10:00:00|      4.5|
|      1|session_start|2026-01-03 09:00:00|2026-01-01 14:30:00|     42.5|
|      2|session_start|2026-01-02 11:00:00|               NULL|     NULL|
+-------+-------------+-------------------+-------------------+---------+

